<a href="https://colab.research.google.com/github/miguelandresgordon/WayneHomeLab/blob/main/notebooks/basic_training_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a microWakeWord Model

This notebook guides you through training a microWakeWord model. Run the cells **in order**.

**Steps:**
1. [Configuration](#configuration) — set all parameters here, once
2. [Installation](#installation)
3. [Sample generation](#sample-generation) — generate and verify TTS samples
4. [Augmentation data](#augmentation-data) — download background noise and RIRs
5. [Feature generation](#feature-generation) — augment and extract spectrograms
6. [Negative datasets](#negative-datasets) — download pre-generated negatives
7. [Training](#training) — train, quantize, and convert the model
8. [Export](#export) — download the `.tflite` file

> **Note:** The model generated may not be production-ready. Experiment with the configuration parameters (especially training steps and class weights) to improve quality.
>
> To use the output in ESPHome, write a model manifest JSON. See the [ESPHome docs](https://esphome.io/components/micro_wake_word) and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

---
## 1. Configuration <a id="configuration"></a>

**Edit this cell before running anything else.** All tunable parameters live here.

In [1]:
# ── Wake word ─────────────────────────────────────────────────────────────────
TARGET_WORD = 'Mariano'   # Phonetic spelling often produces better TTS samples

# ── Sample generation ─────────────────────────────────────────────────────────
NUM_TRAINING_SAMPLES = 18000  # Number of TTS samples for training
SAMPLE_BATCH_SIZE    = 100
SAMPLES_DIR          = 'generated_samples'

# ── Output directories ────────────────────────────────────────────────────────
FEATURES_DIR         = 'generated_augmented_features'
NEGATIVE_DIR         = 'negative_datasets'
TRAIN_DIR            = 'trained_models/wakeword'

# ── Augmentation ──────────────────────────────────────────────────────────────
AUG_DURATION_S       = 3.2
AUG_PROBABILITIES    = {
    'SevenBandParametricEQ': 0.1,
    'TanhDistortion':        0.1,
    'PitchShift':            0.1,
    'BandStopFilter':        0.1,
    'AddColorNoise':         0.1,
    'AddBackgroundNoise':    0.75,
    'Gain':                  1.0,
    'RIR':                   0.5,
}
BG_MIN_SNR_DB        = -5
BG_MAX_SNR_DB        = 10

# ── Training hyperparameters ──────────────────────────────────────────────────
TRAINING_STEPS = [20000, 10000]
LEARNING_RATES = [0.001, 0.0001]
BATCH_SIZE = 64
POSITIVE_CLASS_WEIGHT = [1]
NEGATIVE_CLASS_WEIGHT = [20]
EVAL_STEP_INTERVAL = 500
CLIP_DURATION_MS = 1500

# ── Model selection metric ────────────────────────────────────────────────────
MAXIMIZATION_METRIC = 'average_viable_recall'
MINIMIZATION_METRIC = None
TARGET_MINIMIZATION = 0.9

In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/microwakeword_mariano'
os.makedirs(BASE_DIR, exist_ok=True)

SAMPLES_DIR  = os.path.join(BASE_DIR, 'generated_samples')
FEATURES_DIR = os.path.join(BASE_DIR, 'generated_augmented_features')
NEGATIVE_DIR = os.path.join(BASE_DIR, 'negative_datasets')
TRAIN_DIR    = os.path.join(BASE_DIR, 'trained_models/wakeword')

for d in (SAMPLES_DIR, FEATURES_DIR, NEGATIVE_DIR, TRAIN_DIR):
    os.makedirs(d, exist_ok=True)

print('BASE_DIR =', BASE_DIR)
print('SAMPLES_DIR =', SAMPLES_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR = /content/drive/MyDrive/microwakeword_mariano
SAMPLES_DIR = /content/drive/MyDrive/microwakeword_mariano/generated_samples


---
## 2. Installation <a id="installation"></a>

Run once, then **restart the kernel** before continuing.

In [3]:
import platform, sys, torch

# ── System dependencies ────────────────────────────────────────────────────────
if platform.system() == "Darwin":
    print("macOS: install sox via 'brew install sox' if not already present")
else:
    !apt-get install -y -q sox

# ── Python dependencies ────────────────────────────────────────────────────────
if platform.system() == "Darwin":
    %pip install -q 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

%pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
%pip install -q qwen-tts soundfile ipywidgets

# ── flash-attn (community pre-built wheels — no compilation) ──────────────────
# Source: https://github.com/lesj0610/flash-attention/releases
if torch.cuda.is_available():
    torch_ver  = ".".join(torch.__version__.split(".")[:2])
    python_ver = f"cp{sys.version_info.major}{sys.version_info.minor}"
    wheel = f"flash_attn-2.8.3+cu12torch{torch_ver}cxx11abiTRUE-{python_ver}-{python_ver}-linux_x86_64.whl"
    url   = f"https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch{torch_ver}/{wheel}"
    print(f"Installing flash-attn: {wheel}")
    %pip install -q {url}

# ── microWakeWord ──────────────────────────────────────────────────────────────
!git clone --quiet https://github.com/PrismaKisar/micro-wake-word.git 2>/dev/null || echo "micro-wake-word already cloned"
%pip install -q -e ./micro-wake-word

print("\n✅ Installation complete — restart the kernel now, then run from the Configuration cell.")

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libopencore-amrnb0 libopencore-amrwb0 libsox-fmt-alsa libsox-fmt-base
  libsox3 libwavpack1
Suggested packages:
  libsox-fmt-all
The following NEW packages will be installed:
  libopencore-amrnb0 libopencore-amrwb0 libsox-fmt-alsa libsox-fmt-base
  libsox3 libwavpack1 sox
0 upgraded, 7 newly installed, 0 to remove and 67 not upgraded.
Need to get 617 kB of archives.
After this operation, 1,764 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopencore-amrnb0 amd64 0.1.5-1 [94.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopencore-amrwb0 amd64 0.1.5-1 [49.1 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libsox3 amd64 14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1 [240 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libsox-fmt-alsa

---
## 3. Sample Generation <a id="sample-generation"></a>

In [3]:
# Gate: listen to the previews before proceeding.
# The full sample generation cell depends on _tts_model loaded here.
import torch
import soundfile as sf
import os
from IPython.display import Audio, display

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass  # local run — set HF_TOKEN in your environment if needed

os.makedirs(SAMPLES_DIR, exist_ok=True)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

from qwen_tts import Qwen3TTSModel

load_kwargs = dict(dtype=torch.bfloat16)

if device == "cuda":
    load_kwargs["device_map"] = "auto"
    try:
        import flash_attn
        major, _ = torch.cuda.get_device_capability()
        if major >= 8:
            load_kwargs["attn_implementation"] = "flash_attention_2"
            print("Using flash_attention_2")
    except Exception:
        pass
elif device == "mps":
    load_kwargs["device_map"] = "mps"
else:
    load_kwargs["device_map"] = "cpu"

_tts_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    **load_kwargs,
)

INSTRUCT_CALLING = "Speak naturally as if calling a friend by name"
INSTRUCT_NEUTRAL = "Speak in a clear, neutral tone at a natural pace"
INSTRUCT_DOG     = "Speak as if calling a dog"
MAX_AUDIO_SECS   = 1
MAX_NEW_TOKENS   = 30

for label, instruct, filename in [
    ("Calling (come se chiamassi un amico)", INSTRUCT_CALLING, "preview_calling.wav"),
    ("Neutral (tono neutro e chiaro)",       INSTRUCT_NEUTRAL, "preview_neutral.wav"),
    ("Dog (tono allegro e giocoso)",         INSTRUCT_DOG,     "preview_dog.wav"),
]:
    wavs, sr = _tts_model.generate_custom_voice(
        text=TARGET_WORD,
        language="Spanish",
        speaker="ryan",
        instruct=instruct,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    audio    = wavs[0]
    duration = len(audio) / sr
    if duration > MAX_AUDIO_SECS:
        print(f"⚠️  {filename}: {duration:.1f}s — discarded (hallucination)")
        continue
    path = os.path.join(SAMPLES_DIR, filename)
    sf.write(path, audio, sr)
    print(f"\n▶ {label} ({duration:.2f}s)")
    display(Audio(path, autoplay=False))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



▶ Calling (come se chiamassi un amico) (0.80s)


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


⚠️  preview_neutral.wav: 1.8s — discarded (hallucination)
⚠️  preview_dog.wav: 2.3s — discarded (hallucination)


In [4]:
import os, torch, torchaudio, soundfile as sf
from tqdm import tqdm
from pathlib import Path

TARGET_SR = 16000
BATCH_SIZE = 100
MAX_AUDIO_SECS = 1
MAX_NEW_TOKENS = 30

QWEN_SPEAKERS = ["vivian", "dylan", "eric", "ono_anna", "ryan", "serena", "sohee", "uncle_fu", "aiden"]
INSTRUCTS = [
    "Speak naturally as if calling a friend by name",
    "Speak in a clear, neutral tone at a natural pace",
    "Speak as if calling a dog",
]

os.makedirs(SAMPLES_DIR, exist_ok=True)

# Índice siguiente = máximo número existente + 1 (ignora personal_*.wav y preview)
existing = []
for f in Path(SAMPLES_DIR).glob('*.wav'):
    if f.stem.isdigit():
        existing.append(int(f.stem))
start_idx = (max(existing) + 1) if existing else 0
remaining = NUM_TRAINING_SAMPLES - start_idx
print(f'Ya hay {start_idx} TTS. Faltan {remaining}.')

if remaining <= 0:
    print('Ya están todas. No hace falta regenerar.')
else:
    # Carga el modelo si no está en memoria (tras restart suele no estar)
    if '_tts_model' not in globals():
        from qwen_tts import Qwen3TTSModel
        import torch
        load_kwargs = dict(dtype=torch.bfloat16, device_map='auto')
        try:
            import flash_attn
            if torch.cuda.get_device_capability()[0] >= 8:
                load_kwargs['attn_implementation'] = 'flash_attention_2'
        except Exception:
            pass
        _tts_model = Qwen3TTSModel.from_pretrained(
            'Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice', **load_kwargs
        )

    pairs = []
    for i in range(start_idx, NUM_TRAINING_SAMPLES):
        pairs.append((QWEN_SPEAKERS[i % len(QWEN_SPEAKERS)], INSTRUCTS[i % len(INSTRUCTS)]))

    idx = start_idx
    discarded = 0
    for batch_start in tqdm(range(0, len(pairs), BATCH_SIZE)):
        batch = pairs[batch_start:batch_start + BATCH_SIZE]
        wavs, sr = _tts_model.generate_custom_voice(
            text=[TARGET_WORD] * len(batch),
            language=['Spanish'] * len(batch),
            speaker=[p[0] for p in batch],
            instruct=[p[1] for p in batch],
            max_new_tokens=MAX_NEW_TOKENS,
        )
        for audio in wavs:
            if len(audio) / sr > MAX_AUDIO_SECS:
                discarded += 1
                continue
            if sr != TARGET_SR:
                t = torch.from_numpy(audio).unsqueeze(0).float()
                t = torchaudio.functional.resample(t, sr, TARGET_SR)
                audio = t.squeeze(0).numpy()
            sf.write(os.path.join(SAMPLES_DIR, f'{idx}.wav'), audio, TARGET_SR)
            idx += 1
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f'Listo: {idx} muestras TTS ({discarded} descartadas)')

Ya hay 0 TTS. Faltan 18000.


100%|██████████| 180/180 [1:27:21<00:00, 29.12s/it]

Listo: 8650 muestras TTS (9350 descartadas)


---
## 4. Augmentation Data <a id="augmentation-data"></a>

Downloads background noise and Room Impulse Responses used during augmentation. **This can be slow.**

> The data has mixed licenses — models trained with it should be considered **non-commercial / personal use only**.

In [5]:
import os, scipy, numpy as np, torch, torchaudio
from pathlib import Path
from tqdm import tqdm
from huggingface_hub import snapshot_download

rir_dir = './mit_rirs'
_rir_has_files = os.path.isdir(rir_dir) and any(f.endswith('.wav') for f in os.listdir(rir_dir))
if not _rir_has_files:
    os.makedirs(rir_dir, exist_ok=True)
    cache = snapshot_download('davidscripka/MIT_environmental_impulse_responses', repo_type='dataset')
    audio_files = list(Path(cache).glob('**/*.wav')) + list(Path(cache).glob('**/*.flac'))
    for f in tqdm(audio_files):
        try:
            audio, sr = torchaudio.load(str(f))
            if sr != 16000:
                audio = torchaudio.functional.resample(audio, sr, 16000)
            audio_np = audio.mean(0).numpy()
            scipy.io.wavfile.write(os.path.join(rir_dir, f.stem + '.wav'), 16000, (audio_np * 32767).astype(np.int16))
        except Exception as e:
            print(f"⚠️ Skipping {f.name}: {e}")
    print('✅ MIT RIRs downloaded')
    import shutil
    shutil.rmtree(cache, ignore_errors=True)
    print('   Cache cleaned up')
else:
    print('⏭  MIT RIRs already present, skipping')

Fetching 272 files:   0%|          | 0/272 [00:00<?, ?it/s]

16khz/h002_Bedroom_62txts.wav:   0%|          | 0.00/26.1k [00:00<?, ?B/s]

16khz/h001_Bedroom_65txts.wav:   0%|          | 0.00/23.3k [00:00<?, ?B/s]

16khz/h006_Bedroom_42txts.wav:   0%|          | 0.00/17.4k [00:00<?, ?B/s]

16khz/h005_Office_Small_44txts.wav:   0%|          | 0.00/22.7k [00:00<?, ?B/s]

...

16khz/h270_Hallway_House_1txts.wav:   0%|          | 0.00/43.3k [00:00<?, ?B/s]

16khz/h271_Outside_InTramStopRainShelter(…):   0%|          | 0.00/57.6k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/936 [00:00<?, ?B/s]

100%|██████████| 270/270 [00:01<00:00, 164.98it/s]

✅ MIT RIRs downloaded
   Cache cleaned up


In [6]:
# AudioSet background noise (one shard — download more shards for better quality)
import os, shutil, scipy, numpy as np, torch, torchaudio
from pathlib import Path
from tqdm import tqdm
from huggingface_hub import hf_hub_download, list_repo_files

audioset_16k = './audioset_16k'
_audioset_has_files = os.path.isdir(audioset_16k) and any(Path(audioset_16k).glob('*.wav'))

if _audioset_has_files:
    print('⏭  AudioSet already present, skipping')
else:
    # Download and extract a shard only if not already done
    flac_files = list(Path('audioset/audio').glob('**/*.flac')) if os.path.isdir('audioset/audio') else []
    if not flac_files:
        os.makedirs('audioset', exist_ok=True)
        try:
            all_files = list(list_repo_files('agkphysics/AudioSet', repo_type='dataset'))
            tar_files = sorted(f for f in all_files if f.endswith('.tar'))
            print(f"AudioSet tar files found: {tar_files[:5]}")
            tar_filename = tar_files[-1] if tar_files else None
        except Exception as e:
            print(f"⚠️  Could not list AudioSet files: {e}")
            tar_filename = None

        if tar_filename:
            print(f"Downloading {tar_filename}...")
            tar_path = hf_hub_download(
                repo_id='agkphysics/AudioSet',
                filename=tar_filename,
                repo_type='dataset',
            )
            !tar -xf {tar_path} -C audioset/
            os.remove(tar_path)
            print("✅ AudioSet extracted — tar deleted")
            flac_files = list(Path('audioset/audio').glob('**/*.flac'))
        else:
            print("⚠️  AudioSet non disponibile — verrà usato solo FMA come background noise")

    if flac_files:
        os.makedirs(audioset_16k, exist_ok=True)
        print(f"Converting {len(flac_files)} FLAC files to 16kHz WAV...")
        for f in tqdm(flac_files):
            try:
                audio, sr = torchaudio.load(str(f))
                if sr != 16000:
                    audio = torchaudio.functional.resample(audio, sr, 16000)
                audio_np = audio.mean(0).numpy()
                scipy.io.wavfile.write(os.path.join(audioset_16k, f.stem + '.wav'), 16000, (audio_np * 32767).astype(np.int16))
            except Exception as e:
                print(f"⚠️ Skipping {f.name}: {e}")
        print(f'✅ AudioSet converted ({len(list(Path(audioset_16k).glob("*.wav")))} files)')
        shutil.rmtree("audioset", ignore_errors=True)
        print("   Source files deleted")


AudioSet tar files found: []
⚠️  AudioSet non disponibile — verrà usato solo FMA come background noise


In [7]:
# Free Music Archive (extra-small subset)
import os, shutil, scipy, numpy as np, torch, torchaudio
from pathlib import Path
from tqdm import tqdm

fma_16k = './fma_16k'
_fma_has_files = os.path.isdir(fma_16k) and any(Path(fma_16k).glob('*.wav'))

if _fma_has_files:
    print('⏭  FMA already present, skipping')
else:
    mp3_files = list(Path('fma/fma_small').glob('**/*.mp3')) if os.path.isdir('fma/fma_small') else []
    if not mp3_files:
        os.makedirs('fma', exist_ok=True)
        fname = 'fma_xs.zip'
        !wget -q -O fma/{fname} 'https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/{fname}'
        !cd fma && unzip -q {fname}
        mp3_files = list(Path('fma/fma_small').glob('**/*.mp3'))

    if mp3_files:
        os.makedirs(fma_16k, exist_ok=True)
        for f in tqdm(mp3_files):
            try:
                audio, sr = torchaudio.load(str(f))
                if sr != 16000:
                    audio = torchaudio.functional.resample(audio, sr, 16000)
                audio_np = audio.mean(0).numpy()
                scipy.io.wavfile.write(os.path.join(fma_16k, f.stem + '.wav'), 16000, (audio_np * 32767).astype(np.int16))
            except Exception as e:
                print(f"⚠️ Skipping {f.name}: {e}")
        print('✅ FMA converted')
        shutil.rmtree('fma', ignore_errors=True)
        print('   Source files deleted')


100%|██████████| 210/210 [00:17<00:00, 11.98it/s]

✅ FMA converted
   Source files deleted


---
## 5. Feature Generation <a id="feature-generation"></a>

In [10]:
import os, subprocess, sys, importlib
from pathlib import Path

# Re-install microwakeword if the runtime was restarted and lost the /content/ clone
try:
    import microwakeword
except ModuleNotFoundError:
    if not Path('./micro-wake-word').exists():
        os.system('git clone --quiet https://github.com/PrismaKisar/micro-wake-word.git')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', './micro-wake-word'], check=True)
    importlib.invalidate_caches()
    mww_path = str(Path('./micro-wake-word').resolve())
    if mww_path not in sys.path:
        sys.path.insert(0, mww_path)

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips

def _has_audio(directory):
    return os.path.isdir(directory) and any(Path(directory).glob('*.wav'))

_bg_candidates = ['fma_16k', 'audioset_16k']
_bg_paths      = [p for p in _bg_candidates if _has_audio(p)]
_missing       = [p for p in _bg_candidates if p not in _bg_paths]

if not _bg_paths:
    raise RuntimeError("Nessuna directory di background noise disponibile. Esegui le celle della Sezione 4.")
if _missing:
    print(f"⚠️  Background noise parzialmente mancante: {_missing} — continuo con {_bg_paths}")

clips = Clips(
    input_directory=SAMPLES_DIR,
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=AUG_DURATION_S,
    augmentation_probabilities=AUG_PROBABILITIES,
    impulse_paths=['mit_rirs'],
    background_paths=_bg_paths,
    background_min_snr_db=BG_MIN_SNR_DB,
    background_max_snr_db=BG_MAX_SNR_DB,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)
print(f"✅ Augmenter pronto — background: {_bg_paths}")

⚠️  Background noise parzialmente mancante: ['audioset_16k'] — continuo con ['fma_16k']
✅ Augmenter pronto — background: ['fma_16k']


In [11]:
# Augment a random clip and play it back to verify the pipeline works
from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

augmented = augmenter.augment_clip(clips.get_random_clip())
save_clip(augmented, 'augmented_clip.wav')
Audio('augmented_clip.wav', autoplay=True)

In [8]:
from pathlib import Path
import zipfile

personal_raw = Path(BASE_DIR) / 'personal_samples_raw'
personal_raw.mkdir(parents=True, exist_ok=True)

zip_path = '/content/personal_samples_mariano.zip'  # o ruta en Drive
with zipfile.ZipFile(zip_path) as z:
    z.extractall(personal_raw)

print(len(list(p for p in personal_raw.rglob('*.wav') if '__MACOSX' not in p.parts)))

34


In [ ]:
# Generate augmented spectrogram features for training, validation, and testing
import os
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.spectrograms import SpectrogramGeneration

os.makedirs(FEATURES_DIR, exist_ok=True)

splits = {
    'training':   {'split_name': 'train',      'repeat': 2, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repeat': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repeat': 1, 'slide_frames': 1},
}

for split, cfg in splits.items():
    out_dir = os.path.join(FEATURES_DIR, split)
    os.makedirs(out_dir, exist_ok=True)
    mmap_dir = os.path.join(out_dir, 'wakeword_mmap')
    if os.path.exists(mmap_dir):
        print(f'⏭  {split} features already exist, skipping')
        continue
    print(f'Generating {split} features...')
    spectrograms = SpectrogramGeneration(
        clips=clips, augmenter=augmenter,
        slide_frames=cfg['slide_frames'], step_ms=10,
    )
    RaggedMmap.from_generator(
        out_dir=mmap_dir,
        sample_generator=spectrograms.spectrogram_generator(split=cfg['split_name'], repeat=cfg['repeat']),
        batch_size=100,
        verbose=True,
    )
    print(f'✅ {split} done')

Generating training features...


0it [00:00, ?it/s]

---
## 6. Negative Datasets <a id="negative-datasets"></a>

Pre-generated spectrogram features for negative (non-wake-word) audio. **This can be slow.**

In [ ]:
import os

os.makedirs(NEGATIVE_DIR, exist_ok=True)

link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
filenames  = ['dinner_party.zip', 'no_speech.zip', 'speech.zip']

for fname in filenames:
    zip_path = f'{NEGATIVE_DIR}/{fname}'
    extracted = zip_path.replace('.zip', '')
    if os.path.exists(extracted):
        print(f'⏭  {fname} already extracted, skipping')
        continue
    !wget -O {zip_path} {link_root + fname}
    !unzip -q {zip_path} -d {NEGATIVE_DIR}
    print(f'✅ {fname} ready')

In [ ]:
from pathlib import Path
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
import os

PERSONAL_FEATURES_DIR = os.path.join(BASE_DIR, "personal_augmented_features")
os.makedirs(PERSONAL_FEATURES_DIR, exist_ok=True)

personal_clips = Clips(
    input_directory=str(Path(BASE_DIR) / "personal_samples_raw"),
    file_pattern="*.wav",
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

for split, cfg in {
    "training": {"split_name": "train", "repeat": 40, "slide_frames": 10},
    "validation": {"split_name": "validation", "repeat": 5, "slide_frames": 10},
    "testing": {"split_name": "test", "repeat": 2, "slide_frames": 1},
}.items():
    out_dir = os.path.join(PERSONAL_FEATURES_DIR, split)
    mmap_dir = os.path.join(out_dir, "wakeword_mmap")
    if os.path.exists(mmap_dir):
        print("skip", split)
        continue
    os.makedirs(out_dir, exist_ok=True)
    spectrograms = SpectrogramGeneration(
        clips=personal_clips, augmenter=augmenter,
        slide_frames=cfg["slide_frames"], step_ms=10,
    )
    RaggedMmap.from_generator(
        out_dir=mmap_dir,
        sample_generator=spectrograms.spectrogram_generator(
            split=cfg["split_name"], repeat=cfg["repeat"]
        ),
        batch_size=100, verbose=True,
    )
    print("OK personal", split)

print("PERSONAL_FEATURES_DIR =", PERSONAL_FEATURES_DIR)

---
## 7. Training <a id="training"></a>

In [ ]:
# Build the training config from the Configuration cell and save it to disk
import yaml

config = {
    'window_step_ms':       10,
    'train_dir':            TRAIN_DIR,
    'training_steps':       TRAINING_STEPS,
    'learning_rates':       LEARNING_RATES,
    'batch_size':           BATCH_SIZE,
    'positive_class_weight': POSITIVE_CLASS_WEIGHT,
    'negative_class_weight': NEGATIVE_CLASS_WEIGHT,
    'eval_step_interval':   EVAL_STEP_INTERVAL,
    'clip_duration_ms':     CLIP_DURATION_MS,
    'maximization_metric':  MAXIMIZATION_METRIC,
    'minimization_metric':  MINIMIZATION_METRIC,
    'target_minimization':  TARGET_MINIMIZATION,
    'time_mask_max_size':   [0],
    'time_mask_count':      [0],
    'freq_mask_max_size':   [0],
    'freq_mask_count':      [0],
    'features': [
        {
            'features_dir': PERSONAL_FEATURES_DIR,
            'sampling_weight': 15.0,
            'penalty_weight': 1.0,
            'truth': True,
            'truncation_strategy': 'truncate_start',
            'type': 'mmap',
        },
        {
            'features_dir':       FEATURES_DIR,
            'sampling_weight':    2.0,
            'penalty_weight':     1.0,
            'truth':              True,
            'truncation_strategy':'truncate_start',
            'type':               'mmap',
        },
        {
            'features_dir':       f'{NEGATIVE_DIR}/speech',
            'sampling_weight':    10.0,
            'penalty_weight':     1.0,
            'truth':              False,
            'truncation_strategy':'random',
            'type':               'mmap',
        },
        {
            'features_dir':       f'{NEGATIVE_DIR}/dinner_party',
            'sampling_weight':    10.0,
            'penalty_weight':     1.0,
            'truth':              False,
            'truncation_strategy':'random',
            'type':               'mmap',
        },
        {
            'features_dir':       f'{NEGATIVE_DIR}/no_speech',
            'sampling_weight':    5.0,
            'penalty_weight':     1.0,
            'truth':              False,
            'truncation_strategy':'random',
            'type':               'mmap',
        },
    ],
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)

print('✅ training_parameters.yaml saved')

In [ ]:
# Train, quantize, and convert the model.
# Training resumes automatically if interrupted.
# Change --train 0 to skip training and only convert the best checkpoint.
!python -m microwakeword.model_train_eval \
    --training_config='training_parameters.yaml' \
    --train 1 \
    --restore_checkpoint 1 \
    --test_tf_nonstreaming 0 \
    --test_tflite_nonstreaming 0 \
    --test_tflite_nonstreaming_quantized 0 \
    --test_tflite_streaming 0 \
    --test_tflite_streaming_quantized 1 \
    --use_weights 'best_weights' \
    mixednet \
    --pointwise_filters '64,64,64,64' \
    --repeat_in_block '1, 1, 1, 1' \
    --mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
    --residual_connection '0,0,0,0' \
    --first_conv_filters 32 \
    --first_conv_kernel_size 5 \
    --stride 3

---
## 8. Export <a id="export"></a>

Downloads the quantized streaming `.tflite` model.

To use it in ESPHome, write a model manifest JSON and adjust the probability threshold based on the test results above. You may also need to increase the Tensor Arena size if the model fails to load.

In [ ]:
import os

tflite_path = f'{TRAIN_DIR}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'

if not os.path.exists(tflite_path):
    print(f'Model not found at {tflite_path} — make sure training completed successfully.')
else:
    try:
        from google.colab import files
        files.download(tflite_path)
    except ImportError:
        print(f'Model ready at: {os.path.abspath(tflite_path)}')